# 🎵 Vocal / Voice Remover using Demucs

This notebook uses **[Demucs](https://github.com/facebookresearch/demucs)** by Meta/Facebook Research — a state-of-the-art deep learning model for music source separation.

It separates any song into **4 stems**:
- 🥁 `drums`
- 🎸 `bass`
- 🎹 `other` (instruments)
- 🎤 `vocals`

By discarding the `vocals` stem and recombining the rest, you get an **instrumental/karaoke version**.

### 📁 Expected folder structure
Place all your MP3 files in the **same folder as this notebook**:
```
your_folder/
├── vocal_remover.ipynb   ← this file
├── song01.mp3
├── song02.mp3
├── ...
└── song20.mp3
```
Outputs will be saved to `./separated/` automatically.

## 1. Install dependencies

Run this once. Demucs requires `ffmpeg` for MP3 handling.

In [ ]:
# Install Demucs and audio dependencies
!pip install -q demucs
!pip install -q pydub tqdm

# Check if ffmpeg is available (required for MP3)
import shutil
if shutil.which('ffmpeg'):
    print('✅ ffmpeg found')
else:
    print('⚠️  ffmpeg not found. Install it:')
    print('   macOS:   brew install ffmpeg')
    print('   Ubuntu:  sudo apt install ffmpeg')
    print('   Windows: https://ffmpeg.org/download.html  (add to PATH)')

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# ─── USER SETTINGS ────────────────────────────────────────────────────────────

# Folder containing your MP3 files (default: same folder as this notebook)
INPUT_DIR = Path(".")

# Where to save separated stems
OUTPUT_DIR = Path("./separated")

# Demucs model to use:
#   'htdemucs'        – default, fast, great quality  ✅ recommended
#   'htdemucs_ft'     – fine-tuned, slightly better quality, slower
#   'mdx_extra'       – MDX architecture, excellent vocals isolation
#   'mdx_extra_q'     – MDX quantized, faster but slightly lower quality
MODEL = "htdemucs"

# Set True to also produce a recombined instrumental (no vocals) WAV file
CREATE_INSTRUMENTAL = True

# Use GPU if available (set False to force CPU)
USE_GPU = True

# ──────────────────────────────────────────────────────────────────────────────

# Discover MP3 files
mp3_files = sorted(INPUT_DIR.glob("*.mp3"))
print(f"Found {len(mp3_files)} MP3 file(s):")
for f in mp3_files:
    print(f"  • {f.name}")

if not mp3_files:
    print("\n⚠️  No MP3 files found. Make sure they're in the same folder as this notebook.")

## 3. Check GPU availability

In [ ]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("✅ Apple Silicon MPS detected")
else:
    device = 'cpu'
    print("ℹ️  No GPU found — running on CPU (slower, but works fine)")

if not USE_GPU:
    device = 'cpu'
    print("   USE_GPU=False → forcing CPU")

print(f"\n🖥️  Using device: {device}")

## 4. Run Demucs on all MP3 files

This cell runs the model. Depending on hardware:
- **GPU**: ~1–3 min per song
- **CPU**: ~5–15 min per song

Output stems are saved to `separated/<model>/<song_name>/`.

In [ ]:
import subprocess
import sys
from tqdm.notebook import tqdm

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

failed = []

for mp3 in tqdm(mp3_files, desc="Separating songs"):
    print(f"\n🎵 Processing: {mp3.name}")
    
    cmd = [
        sys.executable, "-m", "demucs",
        "--name", MODEL,
        "--out", str(OUTPUT_DIR),
        "--mp3",          # save stems as mp3 (smaller files)
        str(mp3)
    ]
    
    if device == 'cpu':
        cmd += ["--device", "cpu"]
    elif device == 'mps':
        cmd += ["--device", "mps"]
    # cuda is default when available
    
    result = subprocess.run(cmd, capture_output=False)
    
    if result.returncode != 0:
        print(f"  ❌ Failed: {mp3.name}")
        failed.append(mp3.name)
    else:
        print(f"  ✅ Done: {mp3.name}")

print("\n" + "="*50)
print(f"✅ Succeeded: {len(mp3_files) - len(failed)}/{len(mp3_files)}")
if failed:
    print(f"❌ Failed: {failed}")

## 5. Combine stems into an instrumental (no vocals)

Takes `drums + bass + other` and merges them into a single stereo WAV file — your karaoke/instrumental track.

In [ ]:
import numpy as np
from pydub import AudioSegment

INSTRUMENTAL_DIR = OUTPUT_DIR / "instrumentals"
INSTRUMENTAL_DIR.mkdir(parents=True, exist_ok=True)

stems_to_combine = ["drums", "bass", "other"]  # everything except vocals

if not CREATE_INSTRUMENTAL:
    print("ℹ️  CREATE_INSTRUMENTAL=False, skipping this step.")
else:
    song_dirs = sorted((OUTPUT_DIR / MODEL).iterdir()) if (OUTPUT_DIR / MODEL).exists() else []
    
    if not song_dirs:
        print("⚠️  No separated stems found. Run cell 4 first.")
    
    for song_dir in song_dirs:
        if not song_dir.is_dir():
            continue
        
        song_name = song_dir.name
        print(f"🎹 Building instrumental: {song_name}")
        
        mixed = None
        missing = []
        
        for stem in stems_to_combine:
            # Demucs saves as .mp3 or .wav depending on flags
            stem_path = next(song_dir.glob(f"{stem}.*"), None)
            if stem_path is None:
                missing.append(stem)
                continue
            
            track = AudioSegment.from_file(str(stem_path))
            mixed = track if mixed is None else mixed.overlay(track)
        
        if missing:
            print(f"  ⚠️  Missing stems: {missing}")
        
        if mixed:
            out_path = INSTRUMENTAL_DIR / f"{song_name}_instrumental.mp3"
            mixed.export(str(out_path), format="mp3", bitrate="320k")
            print(f"  ✅ Saved → {out_path}")

print("\nDone! Instrumental files are in:", INSTRUMENTAL_DIR)

## 6. Preview results in-notebook

Listen to any stem (vocals, instrumental, individual tracks) directly here.

In [ ]:
from IPython.display import Audio, display
from ipywidgets import interact, Dropdown

# Collect all output audio files
all_outputs = sorted(OUTPUT_DIR.rglob("*.mp3")) + sorted(OUTPUT_DIR.rglob("*.wav"))

if not all_outputs:
    print("No output files found yet. Run the separation step first.")
else:
    options = {str(p.relative_to(OUTPUT_DIR)): str(p) for p in all_outputs}
    
    @interact(file=Dropdown(options=options, description='Track:'))
    def play_audio(file):
        display(Audio(file, autoplay=False))

## 7. Output file summary

In [ ]:
from pathlib import Path

print("📂 Output directory structure:\n")

def print_tree(directory, prefix=""):
    entries = sorted(Path(directory).iterdir())
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        if entry.is_dir():
            print(f"{prefix}{connector}{entry.name}/")
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, prefix + extension)
        else:
            size_mb = entry.stat().st_size / 1e6
            print(f"{prefix}{connector}{entry.name}  ({size_mb:.1f} MB)")

if OUTPUT_DIR.exists():
    print_tree(OUTPUT_DIR)
else:
    print("No output yet — run the separation step first.")

---
## 📝 Notes & Tips

| Model | Speed | Quality | Notes |
|---|---|---|---|
| `htdemucs` | Fast | ⭐⭐⭐⭐ | Best default choice |
| `htdemucs_ft` | Medium | ⭐⭐⭐⭐½ | Fine-tuned, slightly better |
| `mdx_extra` | Medium | ⭐⭐⭐⭐½ | Excellent vocal isolation |
| `mdx_extra_q` | Fast | ⭐⭐⭐⭐ | Quantized MDX |

**Tips:**
- For **best vocal removal**, try `mdx_extra` — it's specifically tuned for vocal isolation.
- If you run out of GPU memory, add `--segment 7` to the Demucs command to process in smaller chunks.
- All separated stems are full quality; you can use them individually in a DAW.
- Demucs works on WAV, FLAC, OGG, and MP3 — just change `*.mp3` in the glob patterns.